# gemma-4-developer-agent — submission builder

This notebook builds **`/kaggle/working/submission.zip`**, a declarative Google ADK
agent package. The organizers compile it with `adk-submission` and run the resulting
agent on hidden SWE-bench-style tasks with a single permitted model,
`gemma-4-31b-it-qat-w4a16-ct`, on 4x L4 GPUs.

**What the notebook does, and nothing else:**

1. Holds the agent package as in-memory Python string literals.
2. Validates it against every submission constraint the harness enforces.
3. Writes a deterministic zip with `agent.yaml` at the archive root.

**What it deliberately does not do:** no `pip install`, no downloads, no dataset
inputs, no GPU, no network. The only dependencies are the Python standard library
and `PyYAML`, both of which are already present on Kaggle. Every byte it needs is
written inline in the cells below, so the notebook cannot drift from the package
it is supposed to emit.

## Safety design

A notebook that raises an exception scores **zero**, so the build is written as if
it will fail:

- The primary package is validated. If any single check fails, the notebook falls
  back to a **minimal known-good package** (one agent, no sub-agent, short prompt)
  that is designed to have the fewest possible ways to fail on the scorer's machine.
- If even the packaging step fails, a final cell writes an emergency minimal archive
  directly, bypassing the shared packager.
- The last cell asserts the artifact exists and is a readable zip.

## Contents

| Section | Purpose |
|---|---|
| Setup | import the standard library, resolve the working directory |
| Design rationale | why each load-bearing choice is made |
| Library | the static validator and the fallback package |
| Primary package | the agent YAML, prompts and sampling config |
| Validate | run the checks, choose primary or fallback |
| Package | write the zip and a manifest, print a report |


> **Note.** The agent package itself is what scores; this notebook is only the
> mechanism that delivers it. The interesting content is in `prompts/system.md`
> and `sub_agents/code_analyzer.yaml`, reproduced verbatim in the cells below.


In [ ]:
"""Standard-library setup for the submission builder.

Only stdlib plus PyYAML. No pip, no network, no dataset inputs.
"""
import hashlib
import io
import json
import os
import platform
import posixpath
import sys
import textwrap
import traceback
import zipfile
from pathlib import Path

try:
    import yaml
except Exception:  # pragma: no cover - PyYAML ships with Kaggle notebooks
    yaml = None
    print("WARNING: PyYAML is unavailable; structural checks will be skipped.")


def _resolve_working() -> Path:
    """Find the directory the submission zip must be written to.

    Kaggle uses /kaggle/working. Off Kaggle (local verification) fall back to an
    override or a temporary directory, so the notebook is runnable anywhere.
    """
    for candidate in (
        os.environ.get("KAGGLE_WORKING_DIR"),
        "/kaggle/working",
        os.getcwd(),
    ):
        if not candidate:
            continue
        path = Path(candidate)
        if path.is_dir() and os.access(path, os.W_OK):
            return path
    import tempfile

    path = Path(tempfile.mkdtemp(prefix="gemma4_submission_"))
    return path


WORKING = _resolve_working()

print(f"python           {sys.version.split()[0]} ({platform.platform()})")
print(f"working directory {WORKING}")
print(f"pyyaml            {'present' if yaml is not None else 'MISSING'}")
print(f"writable          {os.access(WORKING, os.W_OK)}")


## Why the package looks the way it does

Every choice below has a mechanism in the harness source. Choices justified only
by a leaderboard delta are not made: the public leaderboard covers roughly half of
the hidden tasks, so one task is worth about 0.017 and run-to-run sampling is
stochastic — differences below ~0.05 are noise.

### 1. No LoRA adapter anywhere

The single biggest structural decision. If an adapter is declared, the harness
starts vLLM with 8 LoRA slots at rank 128, which collapses the available KV cache
from roughly 46k tokens to roughly 7.6k. The scoring context is 32,768 tokens, so
that difference is the difference between an agent that can hold a repository in
its head and one that is compacted away constantly. The package declares no
`adapter:` key and ships no `adapters/` directory.

### 2. `eval_config.yaml` — the 12 hour budget is the binding constraint

The whole scoring run gets 12 hours for all ~120 tasks run **sequentially**,
including vLLM startup and verification. That allows roughly six minutes of wall
clock per task, so the per-task agent budget must be much smaller than the cap the
harness would accept.

| Setting | Value | Reason |
|---|---|---|
| `max_time_minutes` | 4 | a measured 12 minute cap exceeded the total compute limit and scored zero |
| `timeout_seconds` | 300 | see below — **not** lowered |
| `max_tool_calls` | 80 | bounds runaway tool loops |
| `max_turns` | 120 | sub-agent LLM calls count against the same budget, and the harness default is a hard-coded 500 |

`timeout_seconds` is widely reported in write-ups as a good way to bound the agent
by lowering it to 120. That is wrong, and dangerously so: it maps to
`harness.command_timeout_seconds`, which bounds **both** the agent's commands
**and** the Phase 2 pytest run. Lowering it can time out the verification of an
already-correct patch, which is strictly worse than a slow agent. It stays at the
300 second default. Agent time is bounded by `max_time_minutes` instead.

### 3. Thinking off, 8192 output tokens

`include_thoughts: false` genuinely hard-disables thinking: it emits
`chat_template_kwargs.enable_thinking=false` and actively removes
`thinking_token_budget` and `reasoning_effort`. `max_output_tokens: 8192` leaves
about 24.5k tokens of prompt room, and reserving more than roughly a third of the
32k context for a single response is harmful. Thinking off also buys more turns per
minute, which matters when the budget is four minutes.

### 4. A coder root agent plus a read-only `code_analyzer` sub-agent

Exploration reads stay out of the coder's context, which is the scarce resource.
`skip_summarization` is **false**: with `true`, the sub-agent's function-response
event becomes final, the parent's flow loop breaks, and the harness immediately
spends a nudge. The sub-agent is read-only and its LLM calls still count against
the parent's budget, which is why `max_turns` is 120 rather than 80.

### 5. Prompt rules that follow from specific harness behaviour

- **Never call `search_similar_code`.** Write-ups claim it returns uncapped source
  and overflows the context. The source says otherwise: it is capped at 5,000
  characters *shared across all ten results*, each head-truncated mid-line. It is
  still banned, but for the honest reason — it burns one of the 80 tool calls to
  return worse results than `git grep`.
- **Never pass `edge_type`.** The shipped graphs contain only lowercase `calls`
  edges, so the documented `CALLS` value matches nothing.
- **Graph tools are weak hints.** They are only populated for the four public
  repositories; hidden tasks come from elsewhere, so they usually just error.
- **Redirect long output to `/tmp`, then `tail` it.** `run_command` keeps only the
  first 5,000 characters of stdout *and* stderr, and pytest prints its verdict at
  the very end — so an unredirected test run hides exactly the part that matters.
- **Use narrow `read_file` ranges.** It returns 150 lines, then 10,000 characters,
  so a window can be clipped mid-line with no warning.
- **Keep a plain-text state card.** Context compaction is active during scoring at
  `token_threshold: 14336`, and only the model's own text survives it. A short
  written card is the only reliable way to keep the required behaviour, the
  confirmed location and the last test result.
- **Have an exact-replacement fallback.** `edit_file` is a three-tier matcher that
  fails on non-unique or reformatted `old_string`. A `str.replace` inside a Python
  heredoc does not have that failure mode.
- **Check `git status --short` before submitting.** The patch is built with
  `git add -N .`, which respects the repository's own `.gitignore`, so a new file
  on an ignored path silently never reaches the grading diff. New files need
  `git add -N -f`. Untracked scratch files inside `/workspace`, on the other hand,
  *do* land in the patch and must be cleaned up.
- **Never edit tests or protected config.** `conftest.py`, `pytest.ini`,
  `pyproject.toml`, `tox.ini`, `setup.cfg`, `sitecustomize.py`, `*.pth`, `test_*.py`,
  `*_test.py` and any `.py` under a `tests`/`test`/`testing` directory are all reset
  to baseline before the hidden `test_patch` is applied. A task whose fix is purely
  a config change is therefore unwinnable by design.
- **Do not rely on `submit_patch` being called.** The harness grades the working
  tree even when it is never called, so a plausible patch beats an empty tree.
- **Both prompts end with `{problem_description}` and contain no other braces.**
  ADK interpolates every brace-delimited token in an instruction from session
  state, so an accidental literal brace is a runtime error. The validator enforces
  that the only placeholders present are `problem_description` and `hints`.

### 6. What is deliberately absent

- **No `skills/` directory.** Skills are progressive disclosure of *content*;
  with unseen target repositories there is no repo-specific content worth
  disclosing, and a `SKILL.md` only adds context.
- **No `.py` files.** `.py` is an allowed extension but is never imported or
  executed by the compiler, so shipping one is pure risk.
- **No files with an empty suffix.** The competition override omits the empty
  extension from the allowlist, so a `LICENSE`, `Makefile`, `Dockerfile` or
  `.gitignore` would abort the entire submission.
- **No callbacks.** `callback_registry` is never passed, so `before_model_callbacks`
  is accepted by the schema and then silently dropped.

### 7. Compile-time correctness

`compile_submission` reports the `code_analyzer` in `tools` rather than in
`sub_agents`, because it is exposed through an inline `agent_tool` mapping. This
is expected and not an error:

```
root 'swe_coder'  model=LiteLlm(model='gemma-4-31b-it-qat-w4a16-ct')  tools=10  output_tokens=8192
```

Ten tools is the nine harness tools plus `code_analyzer`.


## Library and fallback package

The next cell loads the static validator, and the one after that defines the
rollback package.


In [ ]:
# --- validator (spliced verbatim from src/notebook_lib/validator.py) ---------
# The source is embedded as base64 and executed rather than pasted inline: that
# keeps it byte-identical to the repository file, keeps a leading
# ``from __future__ import annotations`` legal, and avoids any escaping problem.
import base64

_SPLICED_SOURCE = '...'

VALIDATOR_SOURCE = base64.b64decode(_SPLICED_SOURCE).decode("utf-8")
VALIDATOR_LOADED = False
try:
    exec(compile(VALIDATOR_SOURCE, "validator.py", "exec"), globals())
    VALIDATOR_LOADED = True
    print(f"validator loaded ({len(VALIDATOR_SOURCE):,d} chars)")
except Exception:  # noqa: BLE001 - never fatal
    print("WARNING: the validator could not be loaded; structural checks are off.")
    print(traceback.format_exc())

if not VALIDATOR_LOADED:
    # Degraded but still functional: packaging proceeds without validation, so a
    # valid archive is produced either way. The constants the reporting cell
    # reads are restated here so no later cell can raise a NameError.
    MODEL = "gemma-4-31b-it-qat-w4a16-ct"
    ALLOWED_SUFFIXES = frozenset(
        {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
    )
    MAX_TOTAL_SIZE_BYTES = 3 * 2**30

    class SubmissionInvalid(Exception):
        """Stand-in used when the real validator could not be imported."""

    def validate(files):
        raise SubmissionInvalid("structural validation is unavailable in this run")

    def pack(files, validate_first=True):
        stream = io.BytesIO()
        with zipfile.ZipFile(stream, "w", zipfile.ZIP_DEFLATED) as archive:
            for name in sorted(files):
                info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                info.create_system = 3
                info.external_attr = 0o644 << 16
                archive.writestr(info, files[name].encode("utf-8"))
        return stream.getvalue()

    def digest(payload):
        return hashlib.sha256(payload).hexdigest()


# --- fallback package -------------------------------------------------------
# A deliberately minimal, deliberately boring package: one agent, no sub-agent,
# no skills, no adapters. This is what ships if the primary package fails any
# check, because a package that compiles is far better than no score at all.


In [ ]:
FALLBACK_FILES = {'agent.yaml': 'name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that fixes one repository issue with a minimal source patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 16384\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 4096\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 300\n  max_time_minutes: 6\n  max_tool_calls: 60\n  max_turns: 120\n', 'prompts/system.md': 'You are an autonomous software engineer fixing one issue in the Python\nrepository at /workspace. Nobody answers questions. Work with your tools until\na fix is in place, then call submit_patch().\n\n## How you are graded\n- Hidden tests are applied to your patch in a fresh checkout. Only source\n  changes count.\n- The working tree is graded even if you never call submit_patch(), so never\n  revert a fix you believe in and never finish with an empty tree.\n- Test files, conftest.py, pytest.ini, pyproject.toml, setup.cfg and tox.ini\n  are reset before grading. Change library code only.\n- Anything left inside /workspace that is not excluded ends up in your patch.\n\n## Environment\n- Offline, dependencies installed. Never run pip install or curl. Never look\n  for source outside /workspace.\n- /workspace is a git repo at the base commit. git, grep, find, sed, awk,\n  head, tail and python3 are available. rg and tree are not.\n- run_command output is cut to its first 5,000 characters, and test runners\n  print their verdict at the end. read_file returns at most 150 lines.\n- read_file, write_file and edit_file accept only paths inside /workspace.\n  Create scratch files in /tmp with run_command.\n- Commands time out after 300 seconds, but a long command only wastes your own\n  budget: never let one run for minutes.\n- get_status() and submit_patch() are free.\n\n## Method\n1. Extract every identifier from the issue: paths, class and function names,\n   error strings, option keys, expected values. Ignore pull-request template\n   text. Write the identifiers down in one short line.\n2. Locate with grep, in one batched command per identifier:\n   git grep -n "IDENTIFIER" -- \'*.py\' | head -30\n   Try a shorter form or the snake_case form if that returns nothing. Never\n   list the whole repository.\n3. Confirm before editing. read_file a narrow line range, or\n   sed -n \'START,ENDp\' FILE. Know the signature, the normal return path and\n   the callers. Do not edit code you have not read.\n4. Fix the root cause minimally. Handle the exact edge cases the issue names,\n   because hidden tests probe them. Keep public signatures and exception types\n   unchanged.\n5. Verify after every edit: python3 -m py_compile PATH\n6. Submit. Run git status --short and git diff --stat, delete debugging files\n   you created in /workspace, then call submit_patch() as your last tool call.\n\n## Long output\nRedirect first, then read the end, in the same command:\n<command> > /tmp/o1.log 2>&1; tail -n 30 /tmp/o1.log\nThen grep -n "Error|FAILED|assert" /tmp/o1.log | head -20 for details.\nUse a new file name each run. /tmp is never part of your patch.\n\n## Editing\nCopy old_string verbatim from the file, including indentation, and keep it\nshort but unique. If edit_file reports more than one match, add surrounding\nlines. If it reports not found, re-read the exact lines and retry once with a\nsmaller snippet. If the same edit fails twice, switch to an exact replacement:\npython3 - << \'PYEOF\'\nfrom pathlib import Path\np = Path("pkg/module.py")\ns = p.read_text()\nold = """<exact text>"""\nnew = """<exact text>"""\nassert s.count(old) == 1, s.count(old)\np.write_text(s.replace(old, new))\nprint("patched")\nPYEOF\nKeep every edit small so the tool call is never cut off. For a genuinely new\nfile use write_file with a path inside /workspace, then py_compile it, then run\ngit status --short and confirm the file is listed: the harness builds the patch\nwith git add -N ., which silently skips anything the repository\'s .gitignore\nignores. If your file is missing, run git add -N -f <path> and check again.\n\n## Tests\nRun only the test that exercises your change:\npython3 -m pytest tests/test_thing.py -q -x -k test_name > /tmp/t1.log 2>&1; tail -n 30 /tmp/t1.log\npython3 -m unittest tests.test_thing > /tmp/t2.log 2>&1; tail -n 30 /tmp/t2.log\nFind candidates with git ls-files | grep test_thing.\nNever run bare pytest, pytest . or unittest discover; a full suite eats your\nwhole budget. Existing failures caused by missing fixtures, missing data files\nor import errors are pre-existing: ignore them, do not fix them, do not edit\ntest code.\n\n## When something fails\nThe same tool call has failed twice: change method instead of retrying. If\nread_file will not return the lines, use sed through run_command. If edit_file\nwill not match, use the exact replacement above. Do not reread a file you\nalready understand.\n\n## Pace\nYou have roughly 4 minutes and 120 model turns. Call get_status() every five or\nsix tool calls; it is free. Explore only until you have a confirmed location,\nthen edit. When half the remaining time is gone, stop exploring, make the most\nplausible minimal fix, run one syntax check and at most one targeted test, and\nsubmit. If a third of the time is gone and you still have no root cause, make\nthe change that best matches the issue wording at the most likely location,\nconfirm it compiles, and submit: a patch that fails some tests scores the same\nas no patch, so shipping a best guess is strictly better than timing out.\n\n## Keeping your place\nThe context is finite and older tool outputs get compacted away as it grows;\nonly your own plain text survives. After confirming the location and after each\nverification, write a state card of at most 100 words: required behaviour,\nconfirmed file and line, edit made, last test result, next single action. Never\npaste tool output into it, and reread the source before editing. Keep tool\nresults small on purpose by piping through head, tail or grep.\n\n## The issue\n{problem_description}\n'}

# Both dictionaries are now plain ``{posix path: file text}`` mappings. The next
# section defines the primary package in the same shape.


## Primary package

The full agent package, inlined as a string mapping. This directory is the single
source of truth in the repository:

```
agent.yaml                  root coder agent: tools + the analyzer sub-agent
eval_config.yaml            the 12 hour compute budget, allocated per task
configs/sampling.yaml       generation config shared by both agents
prompts/system.md           the coder's instructions
prompts/analyzer.md         the sub-agent's instructions
sub_agents/code_analyzer.yaml   read-only navigator
```


In [ ]:
PACKAGE_FILES = {
    'agent.yaml': 'name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that fixes one repository issue with a minimal, verified source patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: false\ngenerate_content_config: !include configs/sampling.yaml',
    'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 16384\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 4096',
    'eval_config.yaml': "# Per-task Stage 1 budgets, read by the scorer from the `evaluation` key.\n#\n# The whole scoring run has a hard 12 hour compute limit covering roughly 120\n# tasks that run one after another, including vLLM start-up, sandbox setup and\n# Phase 2 verification. `max_time_minutes` is a CEILING on the agent loop, not a\n# target: `agent_runner.py` breaks out of the loop the moment `submit_patch` is\n# called, and most tasks finish well before the cap. So a higher ceiling costs\n# nothing on a task the agent solves quickly, and only spends wall clock on the\n# tasks it would otherwise be cut off mid-debug on. A task killed by the timeout\n# with a half-finished edit scores zero, exactly like no patch at all, so the\n# ceiling is where the run risks the 12 hour cap rather than where it loses\n# points. 6 minutes keeps a wide margin under the cap while giving roughly\n# double the exploration, edit and verification time of the previous 4.\n#\n# `timeout_seconds` is deliberately left at the 300 second default. It bounds\n# both the agent's shell commands *and* the Phase 2 pytest run, so lowering it\n# would risk timing out the verification of an otherwise correct patch.\nevaluation:\n  timeout_seconds: 300\n  max_time_minutes: 6\n  max_tool_calls: 60\n  max_turns: 120",
    'prompts/analyzer.md': 'You are `code_analyzer`, a read-only code navigator. You never create, edit or delete a file. The coder agent has asked you a question about the issue reproduced at the bottom of these instructions. Your only job is to tell it exactly where the change belongs and why, so it can act immediately.\n\nYou have a small share of the coder\'s time budget. Answer from the first one or two searches you make. Do not explore exhaustively.\n\n## Tools\n\n- `run_command` for read-only commands: `git grep -n`, `grep -rn`, `sed -n \'START,ENDp\' FILE`, `ls`, `git ls-files \'glob\'`, `wc -l FILE`. Available binaries are `git`, `grep`, `find`, `sed`, `awk`, `head`, `tail`, `python3`. **Not** available: `rg`, `tree`, `jq`. Output is cut to its **first 5,000 characters**, so always pipe through `head` or `grep -n`. Never run a test suite, and never run any command that writes a file.\n- `read_file` with a narrow `start_line`/`end_line` range to confirm what you found. It accepts only paths inside `/workspace`.\n- `get_code_neighbors` and `get_code_subgraph` are available but weak: they show **synchronous** call relationships only and the graph contains no `async def` at all. **Never pass `edge_type`** — the stored value is lowercase `calls`.\n- **Never call `search_similar_code`.** It burns a tool call to return at most about 5,000 characters of code shared across ten results, each truncated mid-line and ranked by an embedding that often returns unrelated code. Plain `git grep` is strictly better.\n- If a tool returns an error, do not retry it. Switch to `git grep`.\n\n## Method\n\n1. Extract the identifiers from the issue: file and module paths, class and function names, error strings, option keys, attribute names, expected values.\n2. Search for each one, most specific first: `git grep -n "EXACT_NAME" -- \'*.py\' | head -30`. Then try a shorter form, the snake_case form of a CamelCase name, or a distinctive substring of the error message.\n3. Open the strongest candidate and read it with `read_file` or `sed`. Follow the code one or two hops until you reach the line whose behaviour diverges from what the issue expects. Check how the function is called so you know the contract you must preserve.\n4. Verify the line numbers by reading them. Never report a line number you have not seen.\n\n## Answer format\n\nReply with exactly these six fields, at most 200 words in total, and nothing else — no preamble, no code blocks, no closing remarks:\n\nLOCATION: `<path>:<start>-<end>` — enclosing function or class\nROOT CAUSE: one or two sentences on what the code does now and what the issue requires instead\nFIX PLAN: the concrete change, naming the exact expression or statement to alter and what it becomes\nRELATED: other files or call sites that need the same change, or `none`\nTESTS: existing test files or `-k` names that exercise this code, or `none`\nCONFIDENCE: `high`, `medium` or `low`\n\nIf you could not confirm a location, set `LOCATION: unconfirmed` and put your best candidate in `FIX PLAN` with `CONFIDENCE: low`. An honest low-confidence report is useful; a wrong confident report makes the coder edit the wrong code.\n\n## The issue\n\n{problem_description}',
    'prompts/system.md': 'You are an autonomous software engineer fixing one issue in the Python repository at `/workspace`. Nobody will answer questions and nobody reviews your work. One goal: leave the smallest correct source change in the working tree, then call `submit_patch`.\n\n## How you are graded\n\n- Hidden tests are applied to your patch in a **fresh checkout**. The grade is binary per task: every hidden test must pass or the task scores zero.\n- **A wrong patch scores exactly the same as no patch — and so does no patch.** There is no penalty for a speculative fix. So never leave the tree empty, never revert a fix you believe in, and never deliberate about whether to ship. Shipping your best guess is free.\n- The working tree is graded **even if you never call `submit_patch`**. Call it anyway, as your last action, because it ends the session immediately.\n- Test files, `conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg` and `tox.ini` are reset to their original state before grading. Editing them can never help. Change library code only.\n- Anything left inside `/workspace` that is not git-ignored ends up in your patch. Untracked scratch files count.\n\n## Environment\n\n- Fully offline; dependencies are installed. **Never** run `pip install`, `curl` or `wget`.\n- `/workspace` is a git repo at the task\'s base commit. `git log`, `git grep`, `git diff`, `git ls-files` are the fastest way to navigate.\n- Available: `git`, `grep`, `find`, `sed`, `awk`, `head`, `tail`, `wc`, `sort`, `cut`, `python3`. **Not** available: `rg`, `tree`, `jq`.\n\n## The four traps that cost points\n\n1. **Output is cut to the FIRST 5,000 characters.** For a test run or a big `grep`, the cut keeps the banner and throws away the verdict, which is always at the END. So a bare `pytest` tells you nothing.\n2. **`read_file` returns at most 150 lines and 10,000 characters.** Read narrow ranges. For a precise window use `sed -n \'START,ENDp\' FILE`, which does not have the 150-line ceiling.\n3. **Only paths inside `/workspace` work** for `read_file`, `write_file` and `edit_file`. `write_file(\'/tmp/x\')` fails. Create scratch files in `/tmp` with a heredoc through `run_command`.\n4. **The harness compacts your context.** After roughly every fifth turn, older tool calls and their outputs are discarded, and only your own plain text survives. Anything you still need must be written to a file.\n\nBecause of trap 4, use a scratchpad:\n\n```\ncat > /tmp/plan.md << \'EOF\'\nISSUE: one line, the behaviour that must change\nFILE: path/to/module.py  LINES: 120-140\nROOT CAUSE: one sentence\nDONE: what you changed, or none\nLAST TEST: pass/fail plus the one-line reason\nNEXT: the single next action\nEOF\n```\n\nWrite it once you have confirmed the location, and **re-read it with `cat /tmp/plan.md` at the start of any turn where you have lost your place.** Update `DONE` and `LAST TEST` after each edit and each test run. `/tmp` is never part of your patch. Do not paste long tool output into it.\n\n## Method\n\n**1. Extract identifiers.** From the issue, pull every concrete name: file paths, module paths, function and class names, CLI flags, option keys, error message strings, attribute names, expected values. Issues are often pull-request descriptions with checklists and templates — ignore the template, keep the substance. Put the one-line requirement in `/tmp/plan.md`.\n\n**2. Locate with one batched grep per identifier.**\n\n```\ngit grep -n "IDENTIFIER" -- \'*.py\' | head -30\ngit grep -l "IDENTIFIER" -- \'*.py\' | head -20\n```\n\nIf nothing matches, try the shorter form (`parse_header` before `parse_header_value`), the snake_case form of a CamelCase name, and a distinctive substring of the error message. If the issue names a file or directory, list it with `git ls-files \'path/glob\' | head -50`.\n\n**Never page through search results.** Do not run the same search with different arguments to see "more". One `-l` listing, then read the two or three best candidates directly. Iterating through every match burns the entire budget and finds nothing extra. Never run `find . -name \'*.py\' | head -500`.\n\n**3. Confirm before editing.** Read the exact line range around the best candidate, with `read_file` or `sed -n`. Read enough to know the function\'s contract: its signature, what it returns normally, and which callers depend on it. **Never edit a line you have not read in this session, and never edit from memory of an old tool result** — that is the single most common cause of a wrong edit.\n\n**4. Fix the root cause, minimally.** Make the smallest change that makes the reported behaviour correct. Handle the exact edge cases the issue names — empty input, `None`, a missing key, a boundary value — because hidden tests usually probe precisely those. Keep public signatures, return types and exception types unchanged unless the issue demands otherwise. If the issue is about a documentation example, the executable source under `docs_src/` is what to fix.\n\n**5. Compile after every single edit. This is not optional.**\n\n```\npython3 -m py_compile path/to/module.py\n```\n\nA file that does not import scores zero, guaranteed. There is no recovery from a syntax error, so pay the one-second check every time. If compilation fails, fix it in the very next call; do not go and explore something else first.\n\n**6. Guard with the repository\'s own tests.** After editing, run the existing tests that plausibly cover the code you changed — the repo\'s own tests, not ones you invent:\n\n```\ngit diff --name-only\ngit ls-files | grep -i "test.*<module>"\npython3 -m pytest tests/test_thing.py -q -x --tb=short 2>&1 | tail -n 30\n```\n\nA self-written proxy test is easy to write so that it passes and proves nothing; the repository\'s existing tests are the honest signal. Use a `/tmp` reproducer only to confirm that the bug reproduced *before* you fixed it, and to confirm the specific behaviour after.\n\n- **Never** run bare `pytest`, `pytest .`, `python3 -m pytest tests/` or `unittest discover`. That burns the whole budget and tells you nothing actionable.\n- Existing tests in these repositories are often already broken for unrelated reasons — missing fixtures, absent data files, no network. **Ignore pre-existing failures.** Do not fix them, do not stub them, do not edit test code, do not create conftest shims. You are only responsible for failures your own change could have caused.\n\n**7. Submit.** `git status --short` and `git diff --stat`, delete anything in `/workspace` you created only for debugging, then call `submit_patch()` as your **last** action and reply with one line saying what changed.\n\n## Recipes\n\nLong or verdict-at-the-end output — always redirect, then read the end:\n\n```\n<command> > /tmp/o1.log 2>&1; tail -n 30 /tmp/o1.log\ngrep -n "Error\\|FAILED\\|assert" /tmp/o1.log | head -20\n```\n\nUse a fresh log name each time so earlier ones survive. Files in `/tmp` are never in your patch.\n\nA reproducer, created with a heredoc because `write_file` cannot reach `/tmp`:\n\n```\ncat > /tmp/repro.py << \'PYEOF\'\nfrom mypkg import thing\nprint("RESULT:", thing("input"))\nPYEOF\npython3 /tmp/repro.py 2>&1 | tail -n 20\n```\n\nAlways print an explicit `RESULT:` marker. A script that prints nothing on success is indistinguishable from one that failed to run.\n\nEditing. Copy `old_string` **verbatim** from the file you just read, including indentation, and keep it short but unique — 3 to 12 lines is ideal. If `edit_file` reports more than one match, add surrounding lines. If it reports "not found", you mistyped or the whitespace differs: re-read the exact lines and retry once with a smaller snippet.\n\nIf `edit_file` fails twice on the same edit, switch to an exact programmatic replacement, which cannot fail for whitespace reasons:\n\n```\npython3 - << \'PYEOF\'\nfrom pathlib import Path\np = Path("pkg/module.py")\ns = p.read_text()\nold = """<exact text>"""\nnew = """<exact text>"""\nassert s.count(old) == 1, s.count(old)\np.write_text(s.replace(old, new))\nprint("patched")\nPYEOF\n```\n\nKeep edits small. One huge `write_file` or `edit_file` can exceed the output limit, get cut off mid tool-call, and waste the turn. For a genuinely new file, use `write_file` with a relative path inside `/workspace`, then `py_compile` it.\n\n**New files can vanish from your patch.** The patch is built with `git add -N .`, and `git add` silently skips anything the repository\'s own `.gitignore` ignores. After creating a file, confirm it appears in `git status --short`; if not, force it with `git add -N -f <path>` and check again.\n\n## Code-graph tools\n\nThe repository may expose `get_code_neighbors`, `get_code_subgraph` and `search_similar_code`. They are weak hints, not a search engine:\n\n- **Never call `search_similar_code`.** One tool call returns at most about 5,000 characters of code shared across ten results, each cut off mid-line, ranked by an embedding that often returns unrelated code. `git grep` plus a targeted read gives you the same code, complete, for the same call.\n- `get_code_neighbors` and `get_code_subgraph` show **synchronous** call relationships only, and the graph contains no `async def` at all, so they are blind to async code. **Never pass `edge_type`** — the stored value is lowercase `calls` and any other value matches nothing.\n- If a graph tool returns an error, stop using it and search the source instead.\n\n## When you are stuck\n\n- The same call has failed twice: **change method, do not retry.** If `read_file` will not give you the lines, use `sed -n \'START,ENDp\'`. If `edit_file` will not match, use the programmatic replacement.\n- If a test fails and you do not understand why, do not blindly rewrite the fix. Re-read the failing assertion, re-read the code, and state one alternative hypothesis in `/tmp/plan.md` before editing again. At most two repair attempts, then ship the better of the two.\n- Do not re-read a file you have already read and understood. Spend turns on new information.\n- Ignore `/tmp`, `.git`, `__pycache__`, `*.egg-info` and `build`; the harness excludes them.\n\n## Pace\n\nYou have roughly six minutes and a few dozen model turns. `get_status()` is free and does not count against your budget — call it every five or six tool calls to see what is left.\n\n- Explore only until you have a confirmed location. Then edit.\n- When half your time is gone, stop exploring for good. Make the most plausible minimal fix, compile it, run at most one targeted test, and submit.\n- If a third of your time is gone and you still have not found the root cause, stop searching. Change the code that best matches the issue\'s wording at the most likely location, confirm it compiles, and submit.\n- **Never end without a change in the tree and without calling `submit_patch()`.**\n\n## The issue you are working on\n\n{problem_description}',
    'sub_agents/code_analyzer.yaml': 'name: code_analyzer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Read-only code navigator. Given the issue and a short question, returns the exact file and line range to change, the root cause, and a concrete fix plan.\ninstruction: !include ../prompts/analyzer.md\ntools:\n  - run_command\n  - read_file\n  - get_code_neighbors\n  - get_code_subgraph\n  - search_similar_code\ngenerate_content_config: !include ../configs/sampling.yaml',
}

In [ ]:
# What the primary package contains, and its total size.
print("primary files:", len(PACKAGE_FILES))
for _path in sorted(PACKAGE_FILES):
    print(f"  {len(PACKAGE_FILES[_path].encode('utf-8')):>8,d} B  {_path}")
print(f"  {sum(len(v.encode('utf-8')) for v in PACKAGE_FILES.values()):>8,d} B  TOTAL")
print()
print("fallback files:", len(FALLBACK_FILES))
for _path in sorted(FALLBACK_FILES):
    print(f"  {len(FALLBACK_FILES[_path].encode('utf-8')):>8,d} B  {_path}")


## Validate, and pick a package

The primary package is checked against every constraint the harness enforces. If
anything fails, the minimal fallback package ships instead. Either way this cell
records which package won and why, and it never raises.


In [ ]:
SELECTED_FILES = None
SELECTED_NAME = None
SELECTION_REASON = None
VALIDATION_REPORT = {}

try:
    VALIDATION_REPORT = validate(PACKAGE_FILES)
    SELECTED_FILES = PACKAGE_FILES
    SELECTED_NAME = "primary"
    SELECTION_REASON = "primary package passed every check"
except SubmissionInvalid as error:
    # A precise, expected failure: the fallback is the designed response.
    SELECTION_REASON = f"primary rejected by the validator: {error}"
except Exception:  # noqa: BLE001 - the notebook must not abort here
    SELECTION_REASON = "primary raised an unexpected error:\n" + traceback.format_exc()

if SELECTED_FILES is None:
    try:
        VALIDATION_REPORT = validate(FALLBACK_FILES)
        SELECTED_FILES = FALLBACK_FILES
        SELECTED_NAME = "fallback"
    except Exception:  # noqa: BLE001
        # Last resort: ship the fallback unvalidated rather than ship nothing.
        SELECTED_FILES = FALLBACK_FILES
        SELECTED_NAME = "fallback (unvalidated)"
        SELECTION_REASON += "\nfallback also failed:\n" + traceback.format_exc()

print(f"selected package : {SELECTED_NAME}")
print(f"reason           : {SELECTION_REASON}")
print()
print("report:")
for _key, _value in sorted(VALIDATION_REPORT.items()):
    print(f"  {_key}: {_value}")


In [ ]:
# Restate the hard submission limits and confirm the selected package satisfies
# them, independently of the validator, so the report is legible to a reviewer.
selected = SELECTED_FILES
selected_bytes = sum(len(v.encode("utf-8")) for v in selected.values())
selected_suffixes = sorted({posixpath.splitext(n)[1].lower() for n in selected})

print()
print("independent confirmation for the selected package:")
print(f"  file count            {len(selected)} (limit 10000)")
print(f"  unpacked bytes        {selected_bytes:,d} (limit 3,221,225,472)")
print(f"  suffixes present      {selected_suffixes}")
print(f"  agent.yaml at root    {'agent.yaml' in selected}")
print(f"  adapters present      {[n for n in selected if n.startswith('adapters/')]}")
print(f"  files with no suffix  {[n for n in selected if not posixpath.splitext(n)[1]]}")

assert "agent.yaml" in selected, "the selected package has no root agent.yaml"
assert all(posixpath.splitext(n)[1].lower() in ALLOWED_SUFFIXES for n in selected), (
    "the selected package contains a file the harness would reject"
)
assert not any(n.startswith("adapters/") for n in selected), (
    "an adapter slipped into the selected package; that collapses the KV cache"
)
assert all(v.strip() for v in selected.values()), "the selected package has an empty file"
assert selected_bytes <= MAX_TOTAL_SIZE_BYTES, "the selected package exceeds 3 GiB"
print()
print("all independent checks passed")


## Package

`pack()` validates once more, then serialises deterministically: files sorted by
path, directory entries omitted, timestamps pinned, so an unchanged package always
produces a byte-identical archive. It also re-reads its own output and checks every
member round-trips exactly, that there are no directory entries, and that
`agent.yaml` sits at the archive root.


In [ ]:
SUBMISSION_PATH = WORKING / "submission.zip"
MANIFEST_PATH = WORKING / "submission_manifest.json"
submission_bytes = None

try:
    submission_bytes = pack(SELECTED_FILES)
except Exception:  # noqa: BLE001 - never let packaging abort the notebook
    print("pack() failed, falling back to a direct minimal archive:")
    print(traceback.format_exc())

if submission_bytes is None:
    # Emergency path: bypass the shared packager entirely. Deterministic, minimal,
    # and known to satisfy every harness limit.
    emergency = {
        "agent.yaml": (
            "name: swe_coder\n"
            "model: gemma-4-31b-it-qat-w4a16-ct\n"
            "description: Autonomous software engineer that fixes one repository issue.\n"
            "instruction: You fix one issue in the Python repository at /workspace. "
            "Work with your tools until a minimal source fix is in place, then call "
            "submit_patch(). The working tree is graded even if you never call it, so "
            "never revert a fix you believe in. Never edit test files or pyproject.toml; "
            "they are reset before grading. Find the cause with git grep, confirm it by "
            "reading a narrow line range, make the smallest edit that fixes the reported "
            "behaviour, check it with python3 -m py_compile, and redirect any long "
            "output to /tmp with run_command so you can tail it. Submit before your "
            "budget runs out; a best guess beats no patch.\n"
            "{problem_description}\n"
        ),
    }
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, "w", zipfile.ZIP_DEFLATED) as archive:
        for _path in sorted(emergency):
            _info = zipfile.ZipInfo(_path, date_time=(1980, 1, 1, 0, 0, 0))
            _info.compress_type = zipfile.ZIP_DEFLATED
            _info.create_system = 3
            _info.external_attr = 0o644 << 16
            archive.writestr(_info, emergency[_path].encode("utf-8"))
    submission_bytes = stream.getvalue()

SUBMISSION_PATH.write_bytes(submission_bytes)
print(f"wrote {SUBMISSION_PATH} ({len(submission_bytes):,d} bytes)")


In [ ]:
# A manifest makes the artifact auditable after the fact: what was chosen, why,
# what is inside, and the digest that pins the exact bytes being submitted.
try:
    with zipfile.ZipFile(SUBMISSION_PATH) as archive:
        members = {}
        for name in archive.namelist():
            raw = archive.read(name)
            members[name] = {
                "bytes": len(raw),
                "sha256": hashlib.sha256(raw).hexdigest(),
            }
        corrupt = archive.testzip()

    manifest = {
        "submission": SUBMISSION_PATH.name,
        "selected_package": SELECTED_NAME,
        "selection_reason": SELECTION_REASON,
        "submission_sha256": digest(submission_bytes),
        "submission_bytes": len(submission_bytes),
        "zip_crc_check": "clean" if corrupt is None else f"corrupt member {corrupt}",
        "members": members,
        "model": MODEL,
        "adapters": [n for n in SELECTED_FILES if n.startswith("adapters/")],
        "unpacked_bytes": sum(len(v.encode("utf-8")) for v in SELECTED_FILES.values()),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {MANIFEST_PATH}")
except Exception:  # noqa: BLE001
    print("manifest could not be written (it is advisory only):")
    print(traceback.format_exc())


In [ ]:
# Final report. Reading the artifact back from disk is the last line of defence:
# these assertions check the file that actually exists, not the bytes in memory.
with zipfile.ZipFile(SUBMISSION_PATH) as archive:
    names = archive.namelist()
    assert archive.testzip() is None, "the written archive is corrupt"
    assert "agent.yaml" in names, "agent.yaml is not at the archive root"
    assert not any(n.endswith("/") for n in names), "the archive has directory entries"
    assert len(names) == len(set(names)), "the archive has duplicate names"
    for name in names:
        assert archive.read(name).decode("utf-8") == SELECTED_FILES[name], (
            f"{name} did not round trip byte-identically"
        )

print()
print("=" * 68)
print(f"  selected package : {SELECTED_NAME}")
print(f"  reason           : {SELECTION_REASON}")
print(f"  archive          : {SUBMISSION_PATH.name}  ({len(submission_bytes):,d} bytes)")
print(f"  sha256           : {digest(submission_bytes)}")
print(f"  members          : {len(names)}")
for name in names:
    print(f"      {name}")
print("=" * 68)
print("submission.zip is written and verified")

# An explicit sentinel, so a log scan can distinguish "finished and verified"
# from "the kernel died part way through" without having to parse the report.
print("DONE_OK")


### Done

`/kaggle/working/submission.zip` holds a declarative ADK agent package that the
organizers can compile. It declares a single model, ships no adapters, respects the
3 GiB and extension limits, and has a valid `agent.yaml` at its root.
